In [ ]:
!pip -q install requests beautifulsoup4 pandas rapidfuzz

In [ ]:
from __future__ import annotations

import re
import json
import time
import html as html_lib
from datetime import datetime, timedelta, date
from typing import Dict, List, Optional, Tuple
from urllib.parse import urljoin, quote

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup, Tag
from rapidfuzz import fuzz
from IPython.display import display, HTML

# =========================
# CONFIG
# =========================
AMC_BASE = "https://www.amctheatres.com"
RT_BASE  = "https://www.rottentomatoes.com"

THEATRES = [
    {"name": "AMC Tustin 14 @ The District",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-tustin-14-at-the-district/showtimes"},
    {"name": "AMC Orange 30",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-orange-30/showtimes"},
    {"name": "AMC Bay Street 16",
     "url": "https://www.amctheatres.com/movie-theatres/showtimes/amc-bay-street-16/showtimes"},
    {"name": "AMC Woodbridge 5",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-woodbridge-5/showtimes"},
]

OVERRIDE_SATURDAY = None
DEBUG_RT = False

# =========================
# REGEX / CONSTANTS
# =========================
SHOWTIME_HREF_RE = re.compile(r"^/showtimes/\d+", re.I)
MOVIE_HREF_RE    = re.compile(r"^/movies/", re.I)
TIME_RE          = re.compile(r"(\d{1,2}:\d{2}\s*[ap]m)", re.I)

RT_TOMA_TXT_1 = re.compile(r"(\d{1,3})\s*[％%]\s*Tomatometer", re.I)
RT_TOMA_TXT_2 = re.compile(r"Tomatometer\s*(\d{1,3})\s*[％%]", re.I)
RT_AUD_TXT_1  = re.compile(r"(\d{1,3})\s*[％%]\s*(Popcornmeter|Audience\s*Score)", re.I)
RT_AUD_TXT_2  = re.compile(r"(Popcornmeter|Audience\s*Score)\s*(\d{1,3})\s*[％%]", re.I)
RT_TOMA_JSON_1 = re.compile(r'"tomatometerScore"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_1  = re.compile(r'"audienceScore"\s*:\s*(\d{1,3})', re.I)

# =========================
# HELPERS (Hardened)
# =========================
def short_fmt(s) -> Optional[str]:
    if s is None or not isinstance(s, str) or pd.isna(s):
        return None
    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= 40 else s[:37] + "..."

def upcoming_weekend_pacific() -> Tuple[date, date]:
    try:
        from zoneinfo import ZoneInfo
        today = datetime.now(ZoneInfo("America/Los_Angeles")).date()
    except: today = date.today()
    
    if OVERRIDE_SATURDAY:
        sat = datetime.strptime(OVERRIDE_SATURDAY, "%Y-%m-%d").date()
    else:
        wd = today.weekday()
        sat = today + timedelta(days=(5-wd)) if wd <= 5 else today + timedelta(days=(12-wd))
    return sat, sat + timedelta(days=1)

def make_session():
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"})
    return s

def fetch_html(session, url, params=None):
    try:
        r = session.get(url, params=params, timeout=20)
        if r.status_code == 200 and "Access Denied" not in r.text: return 200, r.text
    except: pass
    return 0, ""

def extract_time(txt): 
    if not isinstance(txt, str): return None
    m = TIME_RE.search(txt)
    return m.group(1).strip().lower() if m else None

# =========================
# SCRAPER
# =========================
def scrape_amc_for_date(session, name, url, d):
    st, html = fetch_html(session, url, {"date": d.isoformat()})
    if st != 200: return []
    soup = BeautifulSoup(html, "html.parser")
    heads = [(h, h.find("a", href=MOVIE_HREF_RE).get_text(strip=True)) 
             for h in soup.find_all(["h1","h2","h3"]) if h.find("a", href=MOVIE_HREF_RE)]
    
    out = []
    for i, (h, title) in enumerate(heads):
        stop = heads[i+1][0] if i+1 < len(heads) else None
        fmt, el = None, h.next_element
        while el and el is not stop:
            if isinstance(el, Tag):
                txt = el.get_text(" ", strip=True)
                if any(x in txt.lower() for x in ["imax","dolby","laser","reald"]): fmt = txt
                if el.name == "a" and SHOWTIME_HREF_RE.match(el.get("href", "")):
                    t = extract_time(txt)
                    if t: out.append({"movie_title": title, "theatre": name, "show_date": d.isoformat(), "show_time": t, "format_label": fmt})
            el = el.next_element
    return out

def build_showtimes_cell(df_st):
    df_st = df_st.copy().replace({np.nan: None})
    lines = []
    for th in sorted(df_st["theatre"].unique()):
        df_th = df_st[df_st["theatre"] == th]
        lines.append(f"<b>{th}</b>")
        for d_str in sorted(df_th["show_date"].unique()):
            df_d = df_th[df_th["show_date"] == d_str]
            times = [f"{r['show_time']}" + (f" [{short_fmt(r['format_label'])}]" if r['format_label'] else "") 
                     for _, r in df_d.iterrows()]
            lines.append(f"• {d_str}: " + ", ".join(times))
    return "<br>".join(lines)

def rt_get_scores(session, title, cache):
    slug = re.sub(r"[^a-z0-9]+", "_", re.sub(r"['’]", "", title.lower())).strip("_")
    url = f"{RT_BASE}/m/{slug}"
    st, html = fetch_html(session, url)
    if st == 200:
        aud = RT_AUD_JSON_1.search(html)
        crit = RT_TOMA_JSON_1.search(html)
        return (int(aud.group(1)) if aud else None, int(crit.group(1)) if crit else None, url)
    return (None, None, None)

# =========================
# MAIN
# =========================
sat, sun = upcoming_weekend_pacific()
session = make_session()
results = []

for th in THEATRES:
    for d in [sat, sun]:
        results.extend(scrape_amc_for_date(session, th['name'], th['url'], d))
        time.sleep(0.5)

if results:
    df = pd.DataFrame(results)
    movie_list = []
    for title in sorted(df["movie_title"].unique()):
        aud, crit, url = rt_get_scores(session, title, {})
        movie_list.append({"movie_title": title, "RT Aud": aud, "RT Crit": crit, "url": url})
    
    df_m = pd.DataFrame(movie_list)
    agg = pd.DataFrame([{"movie_title": t, "Showtimes": build_showtimes_cell(sub)} 
                        for t, sub in df.groupby("movie_title")])
    
    final = df_m.merge(agg, on="movie_title").sort_values("RT Aud", ascending=False)
    display(HTML(final.to_html(index=False, escape=False)))
else: print("No data found.")